# M04-P01B — Diffusion in Action (GPU)

> ⚠️ **Esta demostración requiere GPU NVIDIA con CUDA.**  
> Está pensada para ejecutarse en **Google Colab con una T4 o superior**. **No forma parte de la ruta CPU de GitHub Codespaces.**

Esta práctica complementa **M04-P01 — VAE vs GAN vs Diffusion**. Allí hemos visto el *forward diffusion* y qué objetivo aprende un denoiser. Aquí utilizaremos un **modelo de difusión latente ya entrenado** para observar una generación real paso a paso.

La idea principal es conectar:

```text
ruido latente
→ denoising iterativo condicionado por texto
→ latentes cada vez más estructurados
→ decoder VAE
→ imagen visible
```

No entrenaremos Stable Diffusion. Lo utilizaremos como desarrolladores de aplicaciones: cargamos un modelo ya entrenado, enviamos un prompt y observamos cómo evoluciona la generación.


## 0. Preparar Google Colab

Antes de ejecutar el notebook:

1. Abre **Entorno de ejecución / Runtime**.
2. Selecciona **Cambiar tipo de entorno de ejecución / Change runtime type**.
3. Elige una **GPU T4** o superior.
4. Guarda el cambio y ejecuta las celdas desde el principio.

La primera carga del modelo puede tardar porque Colab debe descargar los pesos.


In [ ]:
%pip install -q diffusers transformers accelerate safetensors


In [ ]:
import torch
import matplotlib.pyplot as plt
from diffusers import StableDiffusionPipeline

if not torch.cuda.is_available():
    raise RuntimeError(
        "Esta práctica requiere una GPU CUDA. "
        "En Google Colab selecciona Runtime > Change runtime type > T4 GPU y vuelve a ejecutar."
    )

DEVICE = "cuda"
print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())


## 1. Qué modelo vamos a utilizar

Usaremos **Stable Diffusion 1.5** como ejemplo de *latent diffusion*.

No genera directamente todos los píxeles en cada paso. Trabaja principalmente en un **espacio latente comprimido**:

```text
prompt de texto
      ↓
representación de texto
      ↓
latentes inicialmente ruidosos
      ↓
denoising iterativo
      ↓
latentes progresivamente estructurados
      ↓
VAE decoder
      ↓
imagen visible
```

Para poder observar los pasos intermedios, decodificaremos algunos latentes con el VAE del propio pipeline. Por eso las imágenes tempranas son solo una **visualización aproximada del estado latente** en ese momento.

> Importante: esta demo ilustra un modelo de difusión real. No debemos generalizar su arquitectura exacta a todos los modelos de imagen actuales.


In [ ]:
MODEL_ID = "runwayml/stable-diffusion-v1-5"

pipe = StableDiffusionPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    use_safetensors=True,
)
pipe = pipe.to(DEVICE)
pipe.enable_attention_slicing()

print("Modelo:", MODEL_ID)
print("dtype:", pipe.unet.dtype)


## 2. Elegir un prompt

Puedes cambiar `PROMPT_KEY` y volver a ejecutar desde esta celda.

Incluimos tres ejemplos:

- **simple**: composición fácil de reconocer;
- **surreal**: combinación poco habitual para comprobar seguimiento del prompt;
- **enterprise**: una escena relacionada con el caso empresarial del curso.

Stable Diffusion 1.5 suele responder mejor a prompts descriptivos en inglés, aunque la explicación de la práctica sea en español.


In [ ]:
PROMPTS = {
    "simple": (
        "A green sports car at sunset, cinematic photography, "
        "realistic, detailed reflections"
    ),
    "surreal": (
        "A giraffe driving a sports car by the sea, realistic photography, "
        "cinematic light, highly detailed"
    ),
    "enterprise": (
        "A realistic enterprise network operations center during a VPN "
        "authentication incident, large monitoring screens, engineers at work, "
        "cinematic photography, professional corporate environment"
    ),
}

PROMPT_KEY = "enterprise"
prompt = PROMPTS[PROMPT_KEY]

negative_prompt = (
    "low quality, blurry, distorted, deformed, duplicated objects, "
    "unreadable text, cartoon"
)

NUM_STEPS = 40
SNAPSHOT_EVERY = 5
SEED = 42

print("Prompt:", prompt)
print("Pasos:", NUM_STEPS)
print("Semilla:", SEED)


## 3. Abrir la caja: visualizar latentes intermedios

El pipeline va modificando un tensor latente durante cada paso de denoising.

Para hacerlo visible:

1. interceptamos algunos pasos;
2. tomamos los `latents`;
3. deshacemos el factor de escala del VAE;
4. los decodificamos a una imagen RGB;
5. guardamos la imagen para la cuadrícula final.

Esto **no significa** que Stable Diffusion esté generando una imagen completa de píxeles en cada paso. Nosotros estamos usando el decoder para mirar dentro del proceso.


In [ ]:
history = []


def latents_to_image(latents):
    """Decodifica una copia de los latentes para poder visualizarlos."""
    scaling_factor = pipe.vae.config.scaling_factor
    latents_for_decode = latents.detach() / scaling_factor

    with torch.no_grad():
        image = pipe.vae.decode(latents_for_decode, return_dict=False)[0]

    image = (image / 2 + 0.5).clamp(0, 1)
    image = image.cpu().permute(0, 2, 3, 1).float().numpy()
    return image[0]


def callback_visual(pipe, step_index, timestep, callback_kwargs):
    """Guarda una visualización cada ciertos pasos de denoising."""
    if step_index % SNAPSHOT_EVERY == 0:
        image = latents_to_image(callback_kwargs["latents"])
        history.append((step_index, int(timestep), image))
    return callback_kwargs


## 4. Generar la imagen

Fijamos una semilla para que el experimento sea reproducible.

Durante la llamada observaremos estados intermedios, pero la imagen final la produce el pipeline completo después de todos los pasos de denoising.


In [ ]:
history.clear()
generator = torch.Generator(device=DEVICE).manual_seed(SEED)

print("Generando evolución...")

result = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    num_inference_steps=NUM_STEPS,
    generator=generator,
    callback_on_step_end=callback_visual,
    callback_on_step_end_tensor_inputs=["latents"],
)

final_image = result.images[0]
print("Generación terminada.")


In [ ]:
# Añadimos el resultado final como último elemento de la secuencia visual.
items = [(f"Paso {step}", img) for step, timestep, img in history]
items.append((f"Final ({NUM_STEPS} pasos)", final_image))

cols = 4
rows = (len(items) + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(16, 4 * rows))
axes = axes.flatten() if hasattr(axes, "flatten") else [axes]

for ax, (title, img) in zip(axes, items):
    ax.imshow(img)
    ax.set_title(title)
    ax.axis("off")

for ax in axes[len(items):]:
    ax.axis("off")

fig.suptitle(f"Evolución de la difusión — {PROMPT_KEY}", fontsize=15)
plt.tight_layout()
plt.show()


## 5. Cómo interpretar la cuadrícula

No debemos leerla como si el modelo dibujase una imagen pixel a pixel desde izquierda a derecha.

Lo que observamos es:

```text
latentes muy ruidosos
        ↓
primeras estructuras globales
        ↓
composición más estable
        ↓
formas y detalles
        ↓
imagen final
```

En los primeros pasos puede ser difícil reconocer el resultado. Conforme avanza el denoising, el modelo va imponiendo estructura compatible con el prompt.

### Relación con la demostración CPU de M04-P01

En el notebook principal vimos:

```text
x₀ → ruido progresivo → x_t
```

y construimos conceptualmente el objetivo `epsilon` que aprende un denoiser.

Aquí vemos el otro lado con un modelo ya entrenado:

```text
ruido latente
→ denoising iterativo
→ imagen
```

### Lo importante para un desarrollador

No hemos diseñado ni entrenado Stable Diffusion. Hemos integrado un modelo existente mediante una librería y una API Python.

Ese cambio de perspectiva prepara las siguientes prácticas del módulo:

```text
M04-P02 → texto + imagen en Analyze
M04-P03 → generación visual con un foundation model vía API
M04-P04 → integración de capacidades visuales en Enterprise GenAI Assistant
```


## 6. Modifica y observa

Prueba, como mínimo, estas variaciones:

1. cambia `PROMPT_KEY` entre `simple`, `surreal` y `enterprise`;
2. cambia `SEED` manteniendo el mismo prompt;
3. compara `NUM_STEPS = 20` frente a `NUM_STEPS = 40`;
4. observa en qué momento de la cuadrícula empieza a reconocerse la composición global.

Preguntas de reflexión:

- ¿Qué aparece antes: la composición global o los detalles finos?
- ¿Mantener el prompt y cambiar la semilla produce la misma escena?
- ¿Más pasos siempre implican una mejora visual evidente?
- ¿Qué diferencia existe entre entrenar la DCGAN anterior y consumir este modelo ya entrenado?
